In [ ]:
import pandas as pd
import eazy
import os
from astropy.io import fits
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt
from eazy import filters, utils, templates

cat_name = r"C:\Users\maria\OneDrive\Documents\SURF 2026\Code\JADES_fits_data.fits"

with fits.open(cat_name, memmap=True) as hdul:
    
    #hdul.info() # gives you header info, below I printed the the main headers
    #KRON_CONV is the preferred data to use for analysis
    #print("HEADER INFO:\n0 PRIMARY 1 \nFILTERS \n2 FLAG \n3 SIZE \n4 CIRC \n5 CIRC_BSUB \n6 CIRC_CONV \n7 CIRC_CONV_BSUB \n8 KRON \n9 KRON_CONV \n10 MIRI \n11 PHOTOZ \n12 PHOTOZ_KRON \n")
    
    # call the filters and print the table, there are 35 filters
    
    filter_bands = hdul['FILTERS'].data['BAND']
    print(filter_bands)
    print(len(filter_bands))
    FILTERS = Table(hdul['FILTERS'].data)

    redshift = Table(hdul['PHOTOZ_KRON'].data)
    print(redshift['ID','z_a'][23:33])

    
    '''
    for filter in FILTERS:
        print(filter)
    
    #print(FILTERS, '\n') # table not needed here

    # call the main data and print the table
    KRON_CONV = Table(hdul['KRON_CONV'].data)
    print(KRON_CONV)
    
    all_columns = KRON_CONV.colnames

    # Filter for columns that have '_KRON' but do NOT contain '_ei' (or other error flags)
    filter_columns = [col for col in all_columns if '_KRON_S' in col and '_ei' not in col and '_e' not in col and '_bkg' not in col]
    true_filters_KRON = filter_columns[3:]
    for fil in true_filters_KRON:
        print(fil)


    
    print("Column Names:")
    for name in KRON_CONV.colnames:
        print(name)
        # note: F105W doesn't seem to exist on eazy filters or on the page for filters for jwst?
    '''

In [ ]:
#with fits.open(r"C:\Users\maria\miniconda3\envs\eazyenv\Lib\site-packages\eazy\data\eazy-photoz\templates\spline_templates_v2\spline.param.fits", ignore_missing_simple=True) as hdul:
    #print(Table(hdul[1].data))

In [ ]:
# printing out all the filters EAZY recognizes
#res = filters.FilterFile(os.path.join(utils.path_to_eazy_data(),
                         #'eazy-photoz\\filters/FILTER.RES.latest'))

res = filters.FilterFile(os.path.join(utils.path_to_eazy_data())+r'\eazy-photoz\filters\FILTER.RES.latest')
# documentation does not have eazy-photoz as part of the path but it is necessary
print(os.path.join(utils.path_to_eazy_data()))
print(res.NFILT)


# Print out just the JWST filters and their IDs

'''
print(f"{'ID':<6} {'Filter Name':<30}")
print("-" * 40)
for i, f in enumerate(res.filters):
    if 'jwst' in f.name.lower():
        print(f"{(i + 1):<6} {f.name:<40}")
'''
# print all filters

for i in range(res.NFILT):
    print(f'{i+1} {res.filters[i].name}')

In [ ]:
# only essential parameters are set here

import eazy
import eazy.photoz

trans_file = 'jades_trans_file.csv'
jades_table = Table.read(cat_name, hdu='KRON_CONV',format='fits')
params = {}

# 1. Tell it where your data is
params['CATALOG_FILE'] = jades_table
#params['CATALOG_FORMAT'] = 'fits'
params['EXT_NUMBER'] = 9

# 2. Tell it what templates to use to fit the light
params['TEMPLATES_FILE'] = "c:\\Users\\maria\\AppData\\Local\\Programs\\Python\\Python314\\Lib\\site-packages\\eazy\\data\\eazy-photoz\\templates\\spline_templates_v2\\tweak_spline.param"

# 3. Choose a base name for the output files it saves
params['MAIN_OUTPUT_FILE'] = 'jades_run'

# initialize the fitting program
trans_file = 'jades_trans_file.csv'
self = eazy.photoz.PhotoZ(
    param_file=None, # don't call default file
    translate_file=trans_file, # call translate file
    zeropoint_file=None, # no zero points file for now?
    params=params, # call your own params, eazy will use default for those not defined
    load_prior=False, # uh
    load_products=False # uh
    
)


In [ ]:
# fit entire catalog
n_i = 100
n_f = 200

self.set_sys_err(positive=True) # should I change this to False?

# 2. Fit the ENTIRE catalog using 10 of your 16 cores via joblib
chunk_idx = self.idx[n_i:n_f]
self.fit_catalog(chunk_idx, n_proc=10, method='joblib')

In [ ]:
# Derived parameters (z params, RF colors, masses, SFR, etc.)


zout, hdu = self.standard_output(
    simple=False, 
    rf_pad_width=0.5,
    rf_max_err=2, 
    prior=True,
    beta_prior=True, 
    absmag_filters=[], 
    extra_rf_filters=[]
)

# 'zout' also saved to [MAIN_OUTPUT_FILE].zout.fits

In [ ]:
uv = -2.5*np.log10(zout['restU']/zout['restV'])
vj = -2.5*np.log10(zout['restV']/zout['restJ'])
ssfr = zout['sfr']/zout['mass']


sel = (zout['z_phot'] > 0.2) & (zout['z_phot'] < 1)
print(sel)
print(len(vj[sel]))
print(len(uv[sel]))
plt.scatter(vj[sel], uv[sel], c=np.log10(ssfr)[sel], 
            vmin=-13, vmax=-8, alpha=0.5, cmap='RdYlBu')

plt.xlim(-0.2, 2.3); plt.ylim(0, 2.4); plt.grid()
plt.xlabel(r'$(V-J)_0$'); plt.ylabel(r'$(U-V)_0$')  

In [ ]:
# Show brightest objects with z_spec > 1
n_init = 22
n_final = 32
ifilter = self.flux_columns[np.argmin((self.lc - 8140)**2)]

imag = 31.40 - 2.5*np.log10(self.cat[ifilter])
#sel = (self.ZPHOTO > 12)

so = np.argsort(imag[n_init:n_final])
ids = self.OBJID[n_init:n_final][so]

for i in range(10):
    fig, data = self.show_fit(ids[i], xlim=[0.2, 3], show_components=True,
                              logpz=True, zr=[0,4])